# ML-HW10: RAG по фактам о Берлине / Германии

Локальная RAG на данных из `Introduction to RAG.ipynb`: FAISS + **`Qwen/Qwen2.5-1.5B-Instruct`** (вместо GPT-2).
Промпты — под туристические/энциклопедические факты о Берлине.
Обязательный блок: подбор и сравнение **temperature / top_k / top_p** (+ top_k поиска) с метриками и графиками.


## 1. Импорты

In [ ]:
import os
from pathlib import Path

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from berlin_documents import DOCUMENTS
from questions import HOMEWORK_QUESTIONS
from rag_system import (
    CACHE_DIR,
    GEN_MODEL_NAME,
    GenParams,
    ensure_berlin_index,
    rag_answer,
)

print('модель генерации:', GEN_MODEL_NAME)
print('кэш:', CACHE_DIR)
print('документов в датасете:', len(DOCUMENTS))


## 2. Данные

61 факт о Берлине из коллаба. В индексе — исходные английские предложения; вопросы и ответы — на русском (кросс-языковой поиск через multilingual MiniLM).


In [ ]:
print('примеры документов:')
for i, d in enumerate(DOCUMENTS[:5], 1):
    print(f'{i}. {d}')
print('...')
print(f'всего: {len(DOCUMENTS)}')


## 3. FAISS-индекс

In [ ]:
pack = ensure_berlin_index(force=False)
docs = pack['documents']
index = pack['index']
print('ntotal =', index.ntotal)


## 4. Проверочные вопросы

1. Какая столица Германии? → Berlin  
2. Сколько мостов в Берлине? → ~1700  
3. Кто открыл путь в Бразилию? → в базе нет


In [ ]:
QUESTIONS_HW = HOMEWORK_QUESTIONS
for item in QUESTIONS_HW:
    print(item['kind'], ':', item['q'])
    print('  эталон:', item['gold'])


## 5. Базовый RAG

In [ ]:
BASE = GenParams(temperature=0.1, top_k=50, top_p=0.8, max_new_tokens=100)
DISTANCE_THRESHOLD = 40.0

for item in QUESTIONS_HW:
    out = rag_answer(
        item['q'],
        index,
        docs,
        top_k=3,
        distance_threshold=DISTANCE_THRESHOLD,
        params=BASE,
    )
    print('Q:', item['q'])
    print('distances:', [round(x, 2) for x in out['distances']])
    print('relevant:', len(out['relevant']), '| used_llm:', out['used_llm'])
    print('A:', out['answer'])
    print('-' * 60)


## 6. Эксперименты с параметрами

Один параметр за раз. Значения из задания:  
`top_k_search ∈ {1,3,5,10}`, `temperature ∈ {0.1,0.4,0.7,1.0}` (3 повтора),  
`top_k_gen ∈ {5,10,50,100}`, `top_p ∈ {0.3,0.5,0.8,1.0}`.


In [ ]:
local_exp = Path('data') / 'experiments_berlin.csv'
exp_path = local_exp if local_exp.exists() else (CACHE_DIR / 'experiments_berlin.csv')
if not exp_path.exists():
    %run run_experiments.py
    exp_path = CACHE_DIR / 'experiments_berlin.csv'
    local_exp.parent.mkdir(exist_ok=True)
    local_exp.write_bytes(exp_path.read_bytes())
exp_df = pd.read_csv(exp_path)
exp_df['ok'] = exp_df['eval'].astype(str).str.startswith('ok')

def family(s):
    s = str(s)
    if s.startswith('baseline'):
        return 'baseline'
    if 'top_k_search' in s:
        return 'top_k_search'
    if 'temperature' in s:
        return 'temperature'
    if 'top_k_gen' in s:
        return 'top_k_gen'
    if 'top_p' in s:
        return 'top_p'
    return 'other'

exp_df['family'] = exp_df['settings'].map(family)
print('строк:', len(exp_df), '| модель:', exp_df['model'].iloc[0] if len(exp_df) else None)
display(exp_df[['family', 'settings', 'question', 'eval', 'sec', 'answer']].head(12))


### 6.1. Baseline

In [ ]:
base_view = exp_df[exp_df['family'] == 'baseline'][
    ['question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True)
display(base_view)
ok_rate = exp_df.loc[exp_df['family'] == 'baseline', 'ok'].mean()
print(f'baseline ok_rate={ok_rate:.0%} на {len(base_view)} вопросах')


### 6.2. top_k в поиске

In [ ]:
tk = exp_df[exp_df['family'] == 'top_k_search'].copy()
tk['top_k_search'] = tk['settings'].str.extract(r'top_k_search=(\d+)').astype(int)
display(tk.sort_values(['top_k_search', 'question'])[
    ['top_k_search', 'question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True))

rate = tk.groupby('top_k_search')['ok'].mean()
sec = tk.groupby('top_k_search')['sec'].mean()
display(rate.to_frame('ok_rate'))
display(sec.to_frame('mean_sec'))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
rate.plot(kind='bar', ax=ax[0], color='#2f6fed', rot=0)
ax[0].set_ylim(0, 1.05)
ax[0].set_title('ok_rate vs top_k поиска')
ax[0].set_ylabel('ok_rate')
sec.plot(kind='bar', ax=ax[1], color='#e67e22', rot=0)
ax[1].set_title('среднее время vs top_k поиска')
ax[1].set_ylabel('сек')
plt.tight_layout()
plt.show()
print(f'ok_rate по k: {rate.min():.0%}–{rate.max():.0%}; лучший k по ok={int(rate.idxmax())}')


### 6.3. temperature

In [ ]:
tp = exp_df[exp_df['family'] == 'temperature'].copy()
tp['temperature'] = tp['settings'].str.extract(r'temperature=([0-9.]+)').astype(float)
tp['repeat'] = tp['settings'].str.extract(r'repeat=(\d+)').astype(int)
display(tp.sort_values(['temperature', 'repeat'])[
    ['temperature', 'repeat', 'eval', 'sec', 'answer']
].reset_index(drop=True))

uniq = tp.groupby('temperature')['answer'].nunique()
ok_t = tp.groupby('temperature')['ok'].mean()
display(pd.DataFrame({'ok_rate': ok_t, 'unique_answers': uniq}))

tp['answer_len'] = tp['answer'].astype(str).str.len()
len_mean = tp.groupby('temperature')['answer_len'].mean()

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
ok_t.plot(kind='bar', ax=ax[0], color='#2f6fed', rot=0)
ax[0].set_ylim(0, 1.05)
ax[0].set_title('ok_rate vs temperature')
uniq.plot(kind='bar', ax=ax[1], color='#16a085', rot=0)
ax[1].set_title('уникальность (3 повтора)')
ax[1].set_ylabel('уникальных из 3')
len_mean.plot(kind='bar', ax=ax[2], color='#e67e22', rot=0)
ax[2].set_title('средняя длина ответа')
ax[2].set_ylabel('символы')
plt.tight_layout()
plt.show()

for t, g in tp.groupby('temperature'):
    print(f'temp={t}: уникальных {g["answer"].nunique()} из {len(g)}')
    for a in g['answer'].drop_duplicates():
        print('  -', a[:200])


### 6.4. top_k в генерации

In [ ]:
tg = exp_df[exp_df['family'] == 'top_k_gen'].copy()
tg['top_k_gen'] = tg['settings'].str.extract(r'top_k_gen=(\d+)').astype(int)
display(tg.sort_values(['top_k_gen', 'question'])[
    ['top_k_gen', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))

rate = tg.groupby('top_k_gen')['ok'].mean()
uniq = tg.groupby('top_k_gen')['answer'].nunique()
display(pd.DataFrame({'ok_rate': rate, 'unique_answers': uniq}))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
rate.plot(kind='bar', ax=ax[0], color='#2f6fed', rot=0)
ax[0].set_ylim(0, 1.05)
ax[0].set_title('ok_rate vs top_k генерации')
uniq.plot(kind='bar', ax=ax[1], color='#8e44ad', rot=0)
ax[1].set_title('разнообразие формулировок')
plt.tight_layout()
plt.show()
print(f'лучший top_k_gen по ok_rate: {int(rate.idxmax())} ({rate.max():.0%})')


### 6.5. top_p

In [ ]:
pp = exp_df[exp_df['family'] == 'top_p'].copy()
pp['top_p'] = pp['settings'].str.extract(r'top_p=([0-9.]+)').astype(float)
display(pp.sort_values(['top_p', 'question'])[
    ['top_p', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))

rate = pp.groupby('top_p')['ok'].mean()
uniq = pp.groupby('top_p')['answer'].nunique()
display(pd.DataFrame({'ok_rate': rate, 'unique_answers': uniq}))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
rate.plot(kind='bar', ax=ax[0], color='#2f6fed', rot=0)
ax[0].set_ylim(0, 1.05)
ax[0].set_title('ok_rate vs top_p')
uniq.plot(kind='bar', ax=ax[1], color='#c0392b', rot=0)
ax[1].set_title('разнообразие формулировок')
plt.tight_layout()
plt.show()
print(f'лучший top_p по ok_rate: {float(rate.idxmax())} ({rate.max():.0%})')


### 6.6. Сводное сравнение

In [ ]:
summary_rows = []
for fam, g in exp_df.groupby('family'):
    summary_rows.append({
        'family': fam,
        'n': len(g),
        'ok_rate': round(g['ok'].mean(), 3),
        'mean_sec': round(g['sec'].mean(), 2),
        'unique_answers': int(g['answer'].nunique()),
    })
summary = pd.DataFrame(summary_rows).sort_values('family')
display(summary)

fig, ax = plt.subplots(figsize=(8, 3.8))
summary.set_index('family')['ok_rate'].plot(kind='bar', ax=ax, color='#2f6fed', rot=20)
ax.set_ylim(0, 1.05)
ax.set_title('Сводка ok_rate по семействам экспериментов')
ax.set_ylabel('ok_rate')
plt.tight_layout()
plt.show()

best = exp_df.groupby('settings')['ok'].mean().sort_values(ascending=False)
print('топ-5 настроек по ok_rate:')
display(best.head(5).to_frame('ok_rate'))
print('худшие 5:')
display(best.tail(5).to_frame('ok_rate'))


## 7. Выводы

Заполняется после прогона экспериментов (ячейка ниже печатает авто-черновик по цифрам).


In [ ]:
base_ok = exp_df.loc[exp_df['family'] == 'baseline', 'ok'].mean()
tk_ok = exp_df.loc[exp_df['family'] == 'top_k_search'].groupby(
    exp_df.loc[exp_df['family'] == 'top_k_search', 'settings'].str.extract(r'top_k_search=(\d+)')[0].astype(int)
)['ok'].mean() if False else None

tk2 = exp_df[exp_df['family'] == 'top_k_search'].copy()
tk2['v'] = tk2['settings'].str.extract(r'top_k_search=(\d+)').astype(int)
temp2 = exp_df[exp_df['family'] == 'temperature'].copy()
temp2['v'] = temp2['settings'].str.extract(r'temperature=([0-9.]+)').astype(float)
tg2 = exp_df[exp_df['family'] == 'top_k_gen'].copy()
tg2['v'] = tg2['settings'].str.extract(r'top_k_gen=(\d+)').astype(int)
pp2 = exp_df[exp_df['family'] == 'top_p'].copy()
pp2['v'] = pp2['settings'].str.extract(r'top_p=([0-9.]+)').astype(float)

tk_rate = tk2.groupby('v')['ok'].mean()
tk_sec = tk2.groupby('v')['sec'].mean()
temp_rate = temp2.groupby('v')['ok'].mean()
tg_rate = tg2.groupby('v')['ok'].mean()
pp_rate = pp2.groupby('v')['ok'].mean()
u01 = int(temp2.loc[temp2.v == 0.1, 'answer'].nunique())
u10 = int(temp2.loc[temp2.v == 1.0, 'answer'].nunique())

lines = [
    f'Модель генерации — {GEN_MODEL_NAME} (не GPT-2); эмбеддинги — multilingual MiniLM; база — {len(docs)} фактов о Берлине.',
    f'Baseline (top_k=3, temp=0.1, top_k_gen=50, top_p=0.8): ok_rate={base_ok:.0%}.',
    f'top_k поиска: ok {tk_rate.min():.0%}–{tk_rate.max():.0%}, лучший k={int(tk_rate.idxmax())}; '
    f'время растёт с k (среднее {tk_sec.min():.1f}–{tk_sec.max():.1f} с).',
    f'temperature: при низких значениях ответы стабильнее (уникальных при 0.1: {u01}/3; при 1.0: {u10}/3), '
    f'ok {temp_rate.min():.0%}–{temp_rate.max():.0%}.',
    f'top_k генерации: лучший={int(tg_rate.idxmax())} (ok={tg_rate.max():.0%}); '
    f'top_p: лучший={float(pp_rate.idxmax())} (ok={pp_rate.max():.0%}).',
    'На вопросе без ответа в базе («путь в Бразилию») корректный отказ считается успехом; галлюцинации снижают оценку.',
    'Практическая рекомендация для фактологического RAG: низкая temperature (0.1–0.4), top_p≈0.8, top_k_gen≈50, top_k поиска=3.',
]
print('\n'.join(lines))


### Итоговые наблюдения (по прогону)

На трёх проверочных вопросах формальный **ok_rate ≈ 100%** почти при всех настройках: столица → Берлин, мосты → ~1700, Бразилия → отказ. Это ожидаемо для коротких фактов и хорошего retrieval.

Различия видны в **качестве формулировок и стабильности**:
1. **Лучшие настройки** — baseline: `temperature=0.1`, `top_p=0.8`, `top_k_gen=50`, `top_k_search=3`. Ответы короткие и одинаковые при повторах.
2. **Худшие по качеству** — `temperature=1.0`: 3/3 уникальных формулировки, смесь EN/RU и «шум» вроде «кэпитанской городской области», хотя факт «Берлин» ещё угадывается.
3. **top_k поиска**: ok не падает, но время растёт (~5.0 → ~6.8 с), число релевантных документов растёт с k; для фактов хватает k=1–3.
4. **top_k_gen / top_p** на этих вопросах почти не меняют ok_rate; влияние слабее, чем у temperature.
5. **Qwen2.5-1.5B-Instruct** вместо GPT-2 даёт нормальные русские ответы по английскому контексту и корректный отказ на out-of-scope вопрос.
6. Практический вывод: для фактологического RAG важнее релевантный retrieval и низкая temperature, чем тонкая настройка nucleus/top-k sampling.
